In [ ]:
# repository: https://github.com/nendo1120/Lib4CompProg
# 拝借：https://qiita.com/hibit/items/60c5ec2ecc80a32474e3
from sortedcontainers import SortedSet

# 半開区間で区間を管理している [l,r)
class Pairset():
    data = None
    
    def __init__(self) -> None:
        self.data = SortedSet()
        self.data.add((-10**18,-10**18))
        self.data.add((10**18,10**18))
        self.component_cnt = 0 # 要素数
        self.segment_cnt =0 # 区間数
    
    def contains(self,x):
        idx = self.data.bisect_right((x,10**18))-1
        L_start,L_end = self.data[idx]
        return x < L_end
    
    def add(self,x):
        idx = self.data.bisect_right((x,10**18))-1
        L_start,L_end = self.data[idx]
        R_start,R_end = self.data[idx+1]
        if x < L_end:
            return False
        if L_end < x and x+1 < R_start:
            self.data.add((x,x+1))
        elif L_end == x and x+1 < R_start:
            self.data.pop(idx)
            self.data.add((L_start,x+1))
        elif L_end < x and x+1 == R_start:
            self.data.pop(idx+1)
            self.data.add((x,R_end))
        else:
            self.data.pop(idx+1)
            self.data.pop(idx)
            self.data.add((L_start,R_end))
        return True
    
    def mex(self,x):
        idx = self.data.bisect_right((x,10**18))-1
        L_start,L_end = self.data[idx]
        if L_end <= x:
            return x
        else:
            return L_end
        
    def expand(self,x):
        idx = self.data.bisect_right((x,10**18))-1
        L_start,L_end = self.data[idx]
        if L_end <= x:
            return x,x            
        else:
            return L_start, L_end
    
    def remove(self,x):
        idx = self.data.bisect_right((x,10**18))-1
        L_start,L_end = self.data[idx]
        if L_end <= x:
            return False
        self.data.pop(idx)
        if L_start < x:
            self.data.add((L_start,x))
        if x+1 < L_end:
            self.data.add((x+1,L_end))
        return True

    def add_interval(self, xL, xR):
        idx = self.data.bisect_left((xL, xR))
        if idx > 0:
            prev_start, prev_end = self.data[idx - 1]
            if xL <= prev_end:
                xL = prev_start
                xR = max(xR, prev_end)
                self.component_cnt -= prev_end - prev_start
                self.segment_cnt -= 1
                self.data.pop(idx - 1)
                idx -= 1
        while idx < len(self.data):
            next_start, next_end = self.data[idx]
            if next_start > xR:
                break
            xR = max(xR, next_end)
            self.component_cnt -= next_end - next_start
            self.segment_cnt -= 1
            self.data.pop(idx)

        self.data.add((xL, xR))
        self.component_cnt += xR - xL
        self.segment_cnt += 1

    def remove_interval(self, xL, xR):
        idx = self.data.bisect_left((xL, xL))
        if idx > 0:
            L_start, L_end = self.data[idx - 1]
            if L_start <= xL < L_end:
                self.data.pop(idx - 1)
                self.component_cnt -= L_end - L_start
                self.segment_cnt -= 1
                if L_start < xL:
                    self.data.add((L_start, xL))
                    self.component_cnt += xL - L_start
                    self.segment_cnt += 1
                if xR < L_end:
                    self.data.add((xR, L_end))
                    self.component_cnt += L_end - xR
                    self.segment_cnt += 1

        while idx < len(self.data):
            L_start, L_end = self.data[idx]
            if L_start >= xR:
                break
            self.data.pop(idx)
            self.component_cnt -= L_end - L_start
            self.segment_cnt -= 1
            if xR < L_end:
                self.data.add((xR, L_end))
                self.component_cnt += L_end - xR
                self.segment_cnt += 1


